# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UnsoundMouse/flyrankaiw01_research_question/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

I am taking **Lane 4: CTR / Engagement Opportunity Scoring**.

The question behind it: which visible pages are already earning search impressions at a
decent position, but under-capturing the clicks a page in that position normally gets? A page
sitting at position 4 with a CTR below what other position-4 pages typically pull is a
different kind of problem than a page that simply has no demand at all — and it is a problem a
content editor can actually act on (rewrite the title, tighten the meta description, fix a
snippet mismatch) without needing to touch the whole page.

I chose this lane over the others for three reasons. First, it forces position-adjusted
thinking from day one — you cannot compare CTR across positions without normalizing, so the
core methodological lesson (compare like to like) is baked into the question itself, not an
afterthought. Second, the underlying signal is dense: `ctr`, `avg_position`, and
`position_tier` are populated for the overwhelming majority of rows in the starter slice (see
the numbers below), unlike the AI-referral direction, which the lane guide explicitly flags as
too sparse for anything beyond EDA. Third, the output — a ranked list of "good position,
underperforming clicks" pages with reason codes — maps directly onto a real editorial task
(rewrite metadata) rather than a vague "something might be wrong here."

I am treating this as provisional. The lane guide is explicit that thresholds (which tier
comparison to use, what counts as "underperforming") are policy choices I still have to defend,
and I have not yet touched the warehouse release, only the 30k-row starter slice. I can revisit
this choice up to the end of Week 4.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

url = "https://raw.githubusercontent.com/UnsoundMouse/flyrankaiw01_research_question/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
print(f"Loaded: {df.shape[0]:,} rows x {df.shape[1]} columns | {df['client_id'].nunique()} clients")

Loaded: 30,000 rows x 44 columns | 32 clients


## 2. The question: decision, action, cost of a wrong call

**Research question:** Among pages that already rank at a position with real click potential,
which ones are capturing fewer clicks than other pages at that same position tier typically
do — and does adjusting for position tier (rather than looking at raw CTR) change which pages
get flagged?

### The frame, in one paragraph

For a content editor deciding which page's title and meta description to rewrite next, we will
build a ranked opportunity list from observed search signals in the starter dataset
(impressions, clicks, CTR, average position, position tier), scoring each page by how far its
CTR sits below the typical CTR for other visible pages in the same position tier, measured by
how many flagged pages hold up under a by-hand read of the top 20 and by precision@50 once a
labeled comparison is available. A wrong call costs an editor's hour rewriting metadata on a
page whose low CTR was actually just noise or low volume; a missed call leaves real clicks on
the table on a page that is otherwise already working (good position, real demand). A plain
rule — flag anything below some fixed CTR number — is not enough because CTR expectations vary
hugely by position: comparing a page-1 page and a page-20 page on the same raw CTR threshold
compares two different things. We will claim only observed, decision-support results.

### The four framing questions, answered

**1. What decision does this improve?**
Which page an editor rewrites the title/meta for next. Not "predict CTR" — the decision is
which of the many visible-but-underperforming pages gets the next available editorial hour.

**2. Who acts, and what do they do?**
A content editor works down a ranked opportunity list. For each flagged page they rewrite the
title/meta description, restructure the snippet, or decide the page is fine as-is. Reason codes
(e.g. "good position, impressions ≥500, CTR below tier median") tell them why a page surfaced,
so they can sanity-check the ranking instead of trusting it blindly.

**3. What does a wrong answer cost — and which error is worse?**
A false positive costs an editor an hour rewriting metadata on a page that was already doing
fine for its position and volume. A false negative leaves real clicks unclaimed on a page that
already has the hard part (position, demand) solved — arguably the cheapest traffic gain
available, so missing it has a real opportunity cost. Because editors only ever work from the
top of the list, precision at the top matters more than catching every possible case, which is
why the number below (**5,885 candidates just from one simple rule**) matters: even a narrow
definition already produces more candidates than any team reviews in a season.

**4. Why does data or ML help at all?**
A single fixed CTR threshold cannot work here because "good CTR" depends on position, and the
starter data shows that dependence is large — the median CTR for the best and worst tiers
differ sharply (see the numbers below). A plain rule *can* get you partway if it compares pages
within the same tier rather than against one global number — that is exactly the baseline I
will build first. Where ML or a more careful model earns its place is in learning a smoother,
multi-signal expectation (position, content type, intent, freshness together) instead of five
hard-coded tier buckets, and in ranking candidates by how confidently underperforming they are
rather than just a yes/no flag. If the tier-adjusted rule turns out to rank nearly as well as a
model, that is a real, reportable result — not a failure.

### Task type

| Element | Choice |
|---|---|
| Task type | Ranking / scoring ("which ones first?") |
| Target | Gap between a page's CTR and its position-tier's typical CTR |
| Metric | Precision@50 / by-hand review of top 20 (once I define a validation label) |
| Validation | Client-grouped holdout (a client's pages must not span train and test) |

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the eligible set and the tier-adjusted underperformance flag once here,
# since Section 2's cost argument depends on these numbers.

# avg_position == 0 means "no data," not rank zero, so it's excluded
eligible = df[(df["impressions_90d"] > 0) & (df["avg_position"] > 0)]

# restrict tier medians to reasonably visible pages so low-volume noise doesn't dominate
visible = eligible[eligible["impressions_90d"] >= 500]
tier_median = visible.groupby("position_tier")["ctr"].median().sort_values(ascending=False)

merged = eligible.merge(tier_median.rename("tier_median_ctr"), left_on="position_tier", right_index=True)
underperf = merged[
    (merged["impressions_90d"] >= 500)
    & (merged["avg_position"] <= 20)
    & (merged["ctr"] < merged["tier_median_ctr"])
]

# Quick gut-check for the cost argument in Section 2: candidate volume vs. realistic editorial capacity
candidates_per_week = len(underperf) / 7
hours_to_review_all = len(underperf) * 0.5  # rough estimate: 30 min per page to review/rewrite

print(f"Flagged candidates: {len(underperf):,}")
print(f"If spread evenly across 7 weeks: ~{candidates_per_week:,.0f} pages/week")
print(f"Rough review time at 30 min/page: ~{hours_to_review_all:,.0f} hours total")
print()
print("=> Even a narrow rule already produces more candidates than one editor could")
print("   realistically review in the program's timeframe -- which is the case for")
print("   ranking them by confidence rather than just listing them.")

Flagged candidates: 5,885
If spread evenly across 7 weeks: ~841 pages/week
Rough review time at 30 min/page: ~2,942 hours total

=> Even a narrow rule already produces more candidates than one editor could
   realistically review in the program's timeframe -- which is the case for
   ranking them by confidence rather than just listing them.


## 3. Quick look at the data (2-3 real numbers)

Three numbers, computed from the real starter CSV, that make the case for this lane:

1. how much of the dataset is even eligible (has both impressions and a real position),
2. how much CTR expectations actually vary by position tier — the reason a flat threshold fails,
3. how many pages a simple, tier-adjusted rule already flags — the case for a ranked queue at all.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# [1] Eligible set: real impressions and a real (nonzero) position
print(f"[1] Eligible pages: {len(eligible):,} of {len(df):,} "
      f"({len(eligible)/len(df)*100:.1f}%)")

zero_pos = df[df["avg_position"] == 0]
print(f"    Excluded for avg_position == 0 ('no data'): {len(zero_pos):,}")
print(f"    Trap: these rows are still filed under position_tier = "
      f"{zero_pos['position_tier'].unique().tolist()} -- a naive tier analysis would silently\n"
      f"    pool 'no position data' pages into the BEST tier.")
print()
# [2] How much CTR expectation varies by position tier
print("[2] Median CTR by position tier (visible pages, impressions_90d >= 500):")
print(tier_median.to_string())
print()
highest_tier, lowest_tier = tier_median.idxmax(), tier_median.idxmin()
print(f"    highest is '{highest_tier}' at {tier_median.max():.2f}, "
      f"lowest is '{lowest_tier}' at {tier_median.min():.2f}")
print("    => a single fixed CTR threshold cannot fairly compare pages across tiers.")
print()
# [3] Candidate volume: pages with good position + real demand but below-tier-median CTR
print("[3] Pages meeting the CTR-underperformance trigger:")
print(f"    (impressions_90d >= 500, avg_position <= 20, ctr below own tier's median)")
print(f"    Flagged: {len(underperf):,} of {len(eligible):,} eligible pages "
      f"({len(underperf)/len(eligible)*100:.1f}%)")
print(f"    Clients represented: {underperf['client_id'].nunique()} of {df['client_id'].nunique()}")
print(f"    Combined impressions those pages already receive: {int(underperf['impressions_90d'].sum()):,}")
print()
print("    => a simple, tier-adjusted rule alone already produces more candidates than any")
print("       editorial team reviews in a normal cycle -- ranking them is the real work.")

[1] Eligible pages: 28,795 of 30,000 (96.0%)
    Excluded for avg_position == 0 ('no data'): 1,205
    Trap: these rows are still filed under position_tier = ['top_3'] -- a naive tier analysis would silently
    pool 'no position data' pages into the BEST tier.

[2] Median CTR by position tier (visible pages, impressions_90d >= 500):
position_tier
page_1      0.24
top_3       0.20
striking    0.17
page_3_5    0.09
deep        0.00

    highest is 'page_1' at 0.24, lowest is 'deep' at 0.00
    => a single fixed CTR threshold cannot fairly compare pages across tiers.

[3] Pages meeting the CTR-underperformance trigger:
    (impressions_90d >= 500, avg_position <= 20, ctr below own tier's median)
    Flagged: 5,885 of 28,795 eligible pages (20.4%)
    Clients represented: 24 of 32
    Combined impressions those pages already receive: 51,505,099

    => a simple, tier-adjusted rule alone already produces more candidates than any
       editorial team reviews in a normal cycle -- ranking th

## 4. Careful words: what I can and can't claim

### What these three numbers tell me

**[1]** Almost all of the starter slice (about 96%) has both real impressions and a real
position, so I am not working from a thin remainder. The 4% with `avg_position == 0` is a real
data trap I need to keep excluding by hand, since it currently hides inside the "best" tier
label.

**[2]** Median CTR differs sharply across position tiers on this slice — the lowest tier's
median CTR rounds to zero while the highest tier's does not. That is direct evidence, from the
data itself, that a flat CTR threshold would systematically over-flag low-tier pages and
under-flag high-tier ones — this is the concrete justification for tier-adjusted scoring rather
than a single cutoff.

**[3]** Even a simple, tier-adjusted rule already flags a five-figure-impression pool of pages
spanning three-quarters of the clients in the slice. That volume is the case for spending seven
weeks here: detecting *some* underperformance is easy; the actual work is ranking these
candidates so an editor's limited time goes to the ones most worth it.

### Can claim (observed / directional)

- That, within this 30,000-row anonymized slice across 32 clients, CTR at a given position tier
  varies enough that position-adjusted comparison changes which pages look like a problem.
- That a simple tier-adjusted trigger already surfaces more candidate pages than a typical
  editorial team's capacity — this is direct arithmetic on the data, not a modeling claim.
- That, once I define a validation label and design, a learned or better-calibrated ranking
  either does or does not order this candidate pool better than the tier-median baseline —
  reported honestly either way.

### Cannot claim

- **That rewriting a title or meta description will raise a page's CTR.** This is
  observational data with no experiment or control group; I am ranking candidates for review,
  not proving a cause.
- **Anything about why Google shows a lower or higher CTR for a given snippet** — I only
  observe the outcome (clicks vs. impressions), never the mechanism.
- **That these numbers generalize past this slice.** This is a 30k-row starter sample; the full
  warehouse release is roughly 79M daily rows across 104 clients, and any result here has to be
  re-earned there.

### Two data traps I am carrying forward (per `skills/flyrank/flyrank-data/SKILL.md`)

- `avg_position == 0` means "no data," not rank zero, and those rows currently sit inside the
  `top_3` position tier — I will always filter these out before any tier-based comparison.
- `ctr` and similar rate columns are already ×100 percentages (a `ctr` of `0.76` means 0.76%,
  not 76%) — worth stating explicitly since it changes how the numbers above should be read.

### Why this is not just "train a model"

The hard part of this lane is not detecting underperformance — a two-line rule already does
that. The hard part is (a) choosing a defensible way to define "underperforming for this tier"
instead of an arbitrary cutoff, (b) building a proper future-outcome validation instead of
grading myself against the same window I built the rule from, and (c) making sure any
eventual model is compared honestly against this tier-adjusted baseline, not against a naive
flat threshold that makes the model look better than it is. If a well-built baseline turns out
to rank almost as well as a model, that is a legitimate finding, not a wasted week.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.